# Transformer 아키텍처 완전 해부 - 실습 코드 1: Transformer Encoder Block (PyTorch)

- Tutorial ID: `expand-transformer-architecture`
- Tutorial: Transformer 아키텍처 완전 해부
- Section ID: `expand-transformer-architecture-code-1`
- Section: 실습 코드 1: Transformer Encoder Block (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Transformer Encoder Block (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn

class TransformerEncoderBlock(nn.Module):
        def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        # Multi-Head Attention
        self.mha = nn.MultiheadAttention(
                        d_model, num_heads, dropout=dropout, batch_first=True
        )
        # Feed-Forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        # Layer Norm (Pre-LN)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
                self.dropout = nn.Dropout(dropout)

        def forward(self, x, mask=None):
        # Pre-LN + Self-Attention + Residual
        normed = self.ln1(x)
                attn_out, _ = self.mha(normed, normed, normed, attn_mask=mask)
                x = x + self.dropout(attn_out)
        # Pre-LN + FFN + Residual
                x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

# 테스트
block = TransformerEncoderBlock()
x = torch.randn(2, 10, 512)
out = block(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")